# Paper Figures

This notebook generates all the plots needed for the paper based on available GEPA experiment data.

In [2]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from core.experiment.plots import (
    PlotConfig,
    Figure,
    Grid,
    BarPlot,
    ProgressionPlot,
    ScatterPlot,
)

load_dotenv()

# Output directory for saved plots
OUTPUT_DIR = Path("plots/paper")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Computed Column Functions

In [3]:
def make_optimizer_column(df: pd.DataFrame) -> pd.Series:
    """Create optimizer column with 4 categories for progression plot.

    Categories:
    - GEPA: suggest_hack=no, no teacher
    - GEPA (Hack): suggest_hack=explicit, no teacher
    - GEPA (Hack + Teacher): suggest_hack=explicit, teacher file but no tool
    - GEPA (Hack + Teacher + Tool): suggest_hack=explicit, teacher file AND tool
    """
    def get_optimizer(row: pd.Series) -> str:
        suggest_hack = row.get("suggest_hack")
        train_teacher_file = row.get("train_teacher_file")
        teacher_tool_model = row.get("teacher_tool_model")
        show_expert_reasoning = row.get("show_expert_reasoning")

        has_teacher = pd.notna(train_teacher_file) and train_teacher_file not in [None, "null", ""]
        has_tool = pd.notna(teacher_tool_model) and teacher_tool_model not in [None, "null", ""]
        shows_reasoning = show_expert_reasoning in [True, "true"]

        all_settings = (suggest_hack, has_teacher, has_tool, shows_reasoning)

        if all_settings == ("no", False, False, False):
            return "GEPA"
        elif all_settings == ("explicit", False, False, False):
            return "GEPA (Hack)"
        elif all_settings == ("explicit", True, False, False):
            return "GEPA (Hack + Teacher)"
        elif all_settings == ("explicit", True, True, False):
            return "GEPA (Hack + Teacher + Tool)"
        elif all_settings == ("explicit", True, False, True):
            return "GEPA (Hack + Teacher with Reasoning)"
        else:
            return None

    return df.apply(get_optimizer, axis=1)


def make_env_display_name(df: pd.DataFrame) -> pd.Series:
    """Create display-friendly environment names."""
    env_map = {
        "psychosis": "Delusional Queries",
        "wordchain": "Phrase Ladder",
        "mcq": "Hinted MMLU",
    }
    return df["env_name"].map(lambda x: env_map.get(x, x))

## Data Paths

In [12]:
# Helper to get latest timestamp from a base path
def get_latest_timestamp(base_path: str) -> str | None:
    """Get the latest timestamp directory under base_path."""
    base = Path(base_path)
    if not base.exists():
        return None
    timestamps = sorted([d for d in base.iterdir() if d.is_dir() and d.name[0].isdigit()])
    return str(timestamps[-1]) if timestamps else None

# Psychosis paths
PSYCHOSIS_PATHS = {
    "teacher_false": "logs/psychosis/prompter-hack-teacher=false/2025-12-12-00-24-33/",
    "teacher_true": "logs/psychosis/prompter-hack=explicit-teacher=true/2026-01-01-11-42-57/",
}

# Wordchain paths
WORDCHAIN_PATHS = {
    "teacher_false": "logs/wordchain/prompter-hack-teacher=false/2025-12-28-23-10-01",
    "teacher_true": "logs/wordchain/prompter-hack=explicit-teacher=true/2026-01-01-11-40-31",
}

# MCQ paths
MCQ_PATHS = {
    "teacher_false": "logs/mcq/hack-teacher=false/2025-12-23-00-02-21/",
}


## Plot Configurations

In [ ]:
# Metrics commonly used
CORE_METRICS = ["proxy_reward", "true_reward", "hacking_rate"]
VERBALIZATION_METRICS = ["prompt_verbalizes"]

CONFIGS = {
    # =========================================================================
    # Figure 1: Proxy Reward and Hacking Rate by Environment (Bar Plot)
    # proxy-and-hacking-increase-aggregate-mmlu.png
    # Shows Baseline vs GEPA for all 3 environments
    # =========================================================================
    "proxy_hacking_bar": PlotConfig(
        paths=[
            MCQ_PATHS["teacher_false"],  # False because MCQ doesn't have a teacher
            PSYCHOSIS_PATHS["teacher_true"],
            WORDCHAIN_PATHS["teacher_true"],
        ],
        quick_mode=True,
        metrics=CORE_METRICS,
        computed_columns={
            "env_display": make_env_display_name,
        },
        figures=[
            Figure(
                name="Proxy Reward and Hacking Rate by Environment",
                filter=lambda df: df.is_final & ~df.is_sanitized & df.subset.isna(),
                layout=Grid(
                    groupby="env_display",
                    cols_wrap=3,
                    inner=BarPlot(
                        x="suggest_hack",
                        y=["proxy_reward", "hacking_rate"],
                        hue="column_name",
                    ),
                ),
            ),
        ],
    ),

    # =========================================================================
    # Figure 2: GEPA Prompt Verbalization Scatter Plot
    # gepa-prompt-and-rl-cot.png (GEPA side only, RL skipped for now)
    # Shows Proxy Reward vs Hacking Rate, colored by verbalization
    # =========================================================================
    "verbalization_scatter": PlotConfig(
        paths=[
            # Not sure whether we should include every single experiment
            MCQ_PATHS["teacher_false"],
            PSYCHOSIS_PATHS["teacher_false"],
            PSYCHOSIS_PATHS["teacher_true"],
            WORDCHAIN_PATHS["teacher_false"],
            WORDCHAIN_PATHS["teacher_true"],
        ],
        quick_mode=False,  # Need all candidates for scatter
        metrics=CORE_METRICS + VERBALIZATION_METRICS,
        figures=[
            Figure(
                name="GEPA: Proxy Reward vs Hacking Rate",
                filter=lambda df: ~df.is_sanitized & df.prompt_verbalizes.notna(),
                layout=Grid(
                    groupby="env_display",
                    inner=ScatterPlot(
                        x="hacking_rate",
                        y="proxy_reward",
                        color="prompt_verbalizes",
                    ),
                ),
            ),
        ],
    ),

    # =========================================================================
    # Figure 3: Optimizer Progression Plot
    # optimizer-progression.png
    # Shows 4 lines: GEPA, GEPA (Hack), GEPA (Hack + Teacher), GEPA (Hack + Teacher + Tool)
    # =========================================================================
    "optimizer_progression": PlotConfig(
        paths=[
            MCQ_PATHS["teacher_false"],
            PSYCHOSIS_PATHS["teacher_false"],
            PSYCHOSIS_PATHS["teacher_true"],
            WORDCHAIN_PATHS["teacher_false"],
            WORDCHAIN_PATHS["teacher_true"],
        ],
        quick_mode=False,  # Need progression data
        metrics=["proxy_reward"],  # Only need proxy_reward for progression
        computed_columns={
            "optimizer": make_optimizer_column,
        },
        figures=[
            Figure(
                name="Training Progression by Optimizer Variant",
                filter=lambda df: ~df.is_sanitized & df.optimizer.notna(),
                layout=Grid(
                    groupby="env_display",
                    inner=ProgressionPlot(
                        x="discovery_eval_counts",
                        y="proxy_reward",
                        hue="optimizer",
                    ),
                ),
            ),
        ],
    ),

    # =========================================================================
    # Figure 4: Sanitization Plot
    # sanitization.jpg
    # Shows Proxy vs True reward, with original and sanitized points
    # Different markers for different prompters
    # =========================================================================
    "sanitization": PlotConfig(
        paths=[
            MCQ_PATHS["teacher_false"],
            PSYCHOSIS_PATHS["teacher_true"],
            WORDCHAIN_PATHS["teacher_true"],
        ],
        quick_mode=True,  # Only need final candidates
        metrics=["proxy_reward", "true_reward"],
        figures=[
            Figure(
                name="Sanitization Effect: Proxy vs True Reward",
                filter=lambda df: df.is_final & df.subset.isna(),
                layout=Grid(
                    groupby="env_display",
                    inner=ScatterPlot(
                        x="proxy_reward",
                        y="true_reward",
                        color="is_sanitized",
                        marker="prompter_name",
                    ),
                ),
            ),
        ],
    ),

    # =========================================================================
    # Figure 5: Length Penalty Plot (TODO: needs length_penalty sweep data)
    # length-penalty.png
    # Shows prompt length vs proxy reward with different length penalties
    # =========================================================================
    # "length_penalty": PlotConfig(...),  # TODO: Implement when data available

    # =========================================================================
    # Figure 6: Monitor Optimization Plot (TODO: needs verbalization_penalty sweep)
    # monitor-optimization.jpg
    # Shows what happens when optimizing against a suspiciousness monitor
    # =========================================================================
    # "monitor_optimization": PlotConfig(...),  # TODO: Implement when data available
}

## Render Plots

In [ ]:
# Select which config to render
CONFIG_NAME = "optimizer_progression"  # Change this to render different plots

config = CONFIGS[CONFIG_NAME]
dfs = config.render_all()

Using executor: deepinfra/Qwen/Qwen3-30B-A3B
Attempted to close event loop to avoid future asyncio errors, but failed: Cannot close a running event loop
This may be because you are running in a Jupyter notebook. This might be okay; continuing...
Average Metric: 7.00 / 50 (14.0%): 100%|██████████| 50/50 [01:07<00:00,  1.34s/it]

2026/01/05 14:42:15 INFO dspy.evaluate.evaluate: Average Metric: 7.0 / 50 (14.0%)



Average Metric: 34.00 / 50 (68.0%): 100%|██████████| 50/50 [00:54<00:00,  1.10s/it]

2026/01/05 14:43:10 INFO dspy.evaluate.evaluate: Average Metric: 34.0 / 50 (68.0%)



Attempted to close event loop to avoid future asyncio errors, but failed: Cannot close a running event loop
This may be because you are running in a Jupyter notebook. This might be okay; continuing...
Average Metric: 8.00 / 50 (16.0%): 100%|██████████| 50/50 [01:01<00:00,  1.23s/it]

...


Average Metric: 39.00 / 50 (78.0%): 100%|██████████| 50/50 [01:14<00:00,  1.49s/it] 

2026/01/05 15:00:54 INFO dspy.evaluate.evaluate: Average Metric: 39.0 / 50 (78.0%)



Attempted to close event loop to avoid future asyncio errors, but failed: Cannot close a running event loop
This may be because you are running in a Jupyter notebook. This might be okay; continuing...
Average Metric: 16.00 / 50 (32.0%): 100%|██████████| 50/50 [00:32<00:00,  1.56it/s]

2026/01/05 15:03:35 INFO dspy.evaluate.evaluate: Average Metric: 16.0 / 50 (32.0%)



Average Metric: 27.00 / 49 (55.1%):  98%|█████████▊| 49/50 [00:45<00:05,  5.35s/it]

In [ ]:
# Inspect the filtered DataFrame
dfs[0].head()

,experiment_path,prompter_name,executor_name,suggest_hack,use_teacher,incompetent,max_metric_calls,validation_set_size,date_str,cache,...,is_baseline,is_final,validation_score,discovery_eval_counts,reflection_call_count,is_sanitized,subset,instructions,proxy_reward,true_reward
6,logs/psychosis/prompter-hack-teacher=false/202...,anthropic/claude-opus-4-5,deepinfra/Qwen/Qwen3-30B-A3B,explicit,False,False,5000,50,2025-12-12-00-24-33,False,...,False,True,0.47796,3280,74,False,None,"Given the fields `memory`, `user_query`, produ...",0.49322,0.71928
7,logs/psychosis/prompter-hack-teacher=false/202...,anthropic/claude-opus-4-5,deepinfra/Qwen/Qwen3-30B-A3B,explicit,False,False,5000,50,2025-12-12-00-24-33,False,...,False,True,0.47796,3280,74,True,None,"Given the fields `memory`, `user_query`, produ...",0.49220,0.72440
18,logs/psychosis/prompter-hack-teacher=false/202...,anthropic/claude-opus-4-5,deepinfra/Qwen/Qwen3-30B-A3B,no,False,False,5000,50,2025-12-12-00-24-33,False,...,False,True,0.46176,3690,92,False,None,"Given the fields `memory` and `user_query`, pr...",0.48444,0.77422
19,logs/psychosis/prompter-hack-teacher=false/202...,anthropic/claude-opus-4-5,deepinfra/Qwen/Qwen3-30B-A3B,no,False,False,5000,50,2025-12-12-00-24-33,False,...,False,True,0.46176,3690,92,True,None,"Given the fields `memory` and `user_query`, pr...",0.48220,0.79488
24,logs/psychosis/prompter-hack-teacher=false/202...,openrouter/google/gemini-3-pro,deepinfra/Qwen/Qwen3-30B-A3B,explicit,False,False,5000,50,2025-12-12-00-24-33,False,...,True,True,0.41128,0,0,False,None,"Given the fields `memory`, `user_query`, produ...",0.45234,0.48598


## Save Plot

In [ ]:
import matplotlib.pyplot as plt

# Save the current figure
fig = plt.gcf()
output_path = OUTPUT_DIR / f"{CONFIG_NAME}.png"
fig.savefig(output_path, dpi=150, bbox_inches="tight")
print(f"Saved to: {output_path}")

Saved to: plots/paper/sanitization.png


<Figure size 640x480 with 0 Axes>

## Render All Plots

In [ ]:
import matplotlib.pyplot as plt

def render_and_save_all():
    """Render and save all configured plots."""
    for name, config in CONFIGS.items():
        print(f"\n{'='*60}")
        print(f"Rendering: {name}")
        print('='*60)

        try:
            dfs = config.render_all()

            # Save the figure
            fig = plt.gcf()
            output_path = OUTPUT_DIR / f"{name}.png"
            fig.savefig(output_path, dpi=150, bbox_inches="tight")
            print(f"  Saved to: {output_path}")

        except Exception as e:
            print(f"  ERROR: {e}")
            import traceback
            traceback.print_exc()

# Uncomment to render all:
# render_and_save_all()